In [9]:
import pandas as pd

MAF_PATH = "cohortMAF.2026-07-20.maf.gz"
MUT_MATRIX = "mutation_by_sample_matrix.tsv"
TPM_PATH = "02_gene_by_sample_TPM.tsv"
OUT_PATH = "integrated_mutation_expression_matrix.tsv"

In [10]:
mut = pd.read_csv(MUT_MATRIX, sep="\t")
print("mutation matrix shape:", mut.shape)
mut.head()

mutation matrix shape: (115180, 619)


,Gene_Name,Mutation,AminoAcid_Change,TCGA-05-4244-01A-01D-1105-08,TCGA-05-4249-01A-01D-1105-08,TCGA-05-4250-01A-01D-1105-08,TCGA-05-4382-01A-01D-1931-08,TCGA-05-4384-01A-01D-1753-08,TCGA-05-4389-01A-01D-1265-08,TCGA-05-4390-01A-02D-1753-08,...,TCGA-NJ-A4YG-01A-22D-A25L-08,TCGA-NJ-A4YI-01A-11D-A25L-08,TCGA-NJ-A4YP-01A-11D-A25L-08,TCGA-NJ-A4YQ-01A-11D-A25L-08,TCGA-NJ-A55A-01A-11D-A25L-08,TCGA-NJ-A55O-01A-11D-A25L-08,TCGA-NJ-A55R-01A-11D-A25L-08,TCGA-NJ-A7XG-01A-12D-A397-08,TCGA-O1-A52J-01A-11D-A25L-08,TCGA-S2-AA1A-01A-12D-A397-08
0,A1BG,c.243C>G,p.Phe81Leu,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A1BG,c.485C>T,p.Pro162Leu,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A1BG,c.553C>T,p.Arg185Trp,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A1BG,c.91C>A,p.Leu31Met,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A1CF,c.1544C>G,p.Ala515Gly,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
tpm = pd.read_csv(TPM_PATH, sep="\t", index_col="GeneName")
print("TPM table shape:", tpm.shape, "(genes, samples)")

# aggregation: median TPM per gene across all tumour samples
gene_level_tpm = tpm.median(axis=1).rename("GeneLevelTPM")
gene_level_tpm.head()

TPM table shape: (63130, 199) (genes, samples)


GeneName
ENSG00000000003    37.028331
ENSG00000000005     0.000000
ENSG00000000419    58.424949
ENSG00000000457    10.122700
ENSG00000000460     8.856298
Name: GeneLevelTPM, dtype: float64

In [12]:
# build Hugo -> Ensembl mapping directly from the MAF (columns Hugo_Symbol and Gene)
mapping = pd.read_csv(MAF_PATH, sep="\t", usecols=["Hugo_Symbol", "Gene"], low_memory=False)
# one Ensembl ID per Hugo symbol (take the most frequent one if MAF lists several)
mapping = (
    mapping.dropna()
    .groupby("Hugo_Symbol")["Gene"]
    .agg(lambda s: s.mode().iat[0])
)
print("unique Hugo symbols in mapping:", len(mapping))
mapping.head()

unique Hugo symbols in mapping: 17704


Hugo_Symbol
A1BG       ENSG00000121410
A1CF       ENSG00000148584
A2M        ENSG00000175899
A2ML1      ENSG00000166535
A3GALT2    ENSG00000184389
Name: Gene, dtype: str

In [13]:
# attach Ensembl ID -> GeneLevelTPM to every mutation row
mut["Ensembl"] = mut["Gene_Name"].map(mapping)
mut["GeneLevelTPM"] = mut["Ensembl"].map(gene_level_tpm)

# put GeneLevelTPM right after AminoAcid_Change, drop helper Ensembl column
sample_cols = [c for c in mut.columns if c.startswith("TCGA-")]
result = mut[["Gene_Name", "Mutation", "AminoAcid_Change", "GeneLevelTPM", *sample_cols]]

print("integrated matrix shape:", result.shape)
print("rows with missing TPM:", result["GeneLevelTPM"].isna().sum())
result.head()

integrated matrix shape: (115180, 620)
rows with missing TPM: 1313


/var/folders/nb/thlknwq90kdf50gvsh_ny8680000gn/T/ipykernel_56299/3849211607.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mut["Ensembl"] = mut["Gene_Name"].map(mapping)
/var/folders/nb/thlknwq90kdf50gvsh_ny8680000gn/T/ipykernel_56299/3849211607.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mut["GeneLevelTPM"] = mut["Ensembl"].map(gene_level_tpm)


,Gene_Name,Mutation,AminoAcid_Change,GeneLevelTPM,TCGA-05-4244-01A-01D-1105-08,TCGA-05-4249-01A-01D-1105-08,TCGA-05-4250-01A-01D-1105-08,TCGA-05-4382-01A-01D-1931-08,TCGA-05-4384-01A-01D-1753-08,TCGA-05-4389-01A-01D-1265-08,...,TCGA-NJ-A4YG-01A-22D-A25L-08,TCGA-NJ-A4YI-01A-11D-A25L-08,TCGA-NJ-A4YP-01A-11D-A25L-08,TCGA-NJ-A4YQ-01A-11D-A25L-08,TCGA-NJ-A55A-01A-11D-A25L-08,TCGA-NJ-A55O-01A-11D-A25L-08,TCGA-NJ-A55R-01A-11D-A25L-08,TCGA-NJ-A7XG-01A-12D-A397-08,TCGA-O1-A52J-01A-11D-A25L-08,TCGA-S2-AA1A-01A-12D-A397-08
0,A1BG,c.243C>G,p.Phe81Leu,1.798816,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A1BG,c.485C>T,p.Pro162Leu,1.798816,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A1BG,c.553C>T,p.Arg185Trp,1.798816,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A1BG,c.91C>A,p.Leu31Met,1.798816,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A1CF,c.1544C>G,p.Ala515Gly,0.000000,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
# save integrated matrix as tab-delimited file
result.to_csv(OUT_PATH, sep="\t", index=False)
print("saved to:", OUT_PATH)

saved to: integrated_mutation_expression_matrix.tsv


In [15]:
# sanity check: GeneLevelTPM for well-known LUAD drivers and the most frequent mutations
result.assign(freq=result[sample_cols].sum(axis=1)) \
      .nlargest(10, "freq")[["Gene_Name", "Mutation", "AminoAcid_Change", "GeneLevelTPM", "freq"]]

/var/folders/nb/thlknwq90kdf50gvsh_ny8680000gn/T/ipykernel_56299/1063662932.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result.assign(freq=result[sample_cols].sum(axis=1)) \


,Gene_Name,Mutation,AminoAcid_Change,GeneLevelTPM,freq
49910,KRAS,c.34G>T,p.Gly12Cys,26.606286,59
49913,KRAS,c.35G>T,p.Gly12Val,26.606286,46
28841,EGFR,c.2573T>G,p.Leu858Arg,23.677444,23
49911,KRAS,c.35G>A,p.Gly12Asp,26.606286,18
49912,KRAS,c.35G>C,p.Gly12Ala,26.606286,17
10856,BRAF,c.1919T>A,p.Val640Glu,13.607035,13
102469,TP53,c.329G>T,p.Arg110Leu,25.748722,8
14693,CCT6B,c.1102A>T,p.Asn368Tyr,1.554104,7
19778,COL14A1,c.161G>C,p.Arg54Thr,14.601929,7
28837,EGFR,c.2303G>T,p.Ser768Ile,23.677444,7
